<a href="https://colab.research.google.com/github/Saisneha0209/Own-Practise/blob/main/Self_HI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from dataclasses import dataclass
from typing import List, Dict
import random
import statistics

@dataclass
class Event:
    event_id: int
    source: str
    value: float
    previous_value: float
    context: str

    important: bool = False

    @property
    def novelty(self):
        return min(abs(self.value - self.previous_value), 1.0)

    @property
    def magnitude(self):
        return min(abs(self.value), 1.0)


class FixedFilter:

    def __init__(self, processing_budget=20):
        self.processing_budget = processing_budget

    def score(self, event):

        return (
            0.50 * event.novelty +
            0.50 * event.magnitude
        )

    def select(self, events):

        ranked = sorted(
            events,
            key=self.score,
            reverse=True
        )

        return ranked[:self.processing_budget]



class HumanoidFilter:

    def __init__(
        self,
        processing_budget=20,
        recheck_fraction=0.30
    ):

        self.processing_budget = processing_budget
        self.recheck_fraction = recheck_fraction

        self.active_context = "navigation"

        # Explicit symbolic priorities.
        # These are NOT ML weights.
        self.source_priority = {
            "vision": 0.90,
            "audio": 0.70,
            "motion": 1.00,
            "temperature": 0.50
        }

    # --------------------------------------------------------
    # INITIAL ATTENTION
    # --------------------------------------------------------

    def first_pass_score(self, event):

        novelty = event.novelty
        magnitude = event.magnitude

        priority = self.source_priority.get(
            event.source,
            0.50
        )

        if event.context == self.active_context:
            context_relevance = 1.0
        else:
            context_relevance = 0.25

        score = (
            0.40 * novelty +
            0.30 * magnitude +
            0.20 * priority +
            0.10 * context_relevance
        )

        return score

    # --------------------------------------------------------
    # DELIBERATE RECHECKING
    # --------------------------------------------------------

    def recheck_score(self, event, events):

        original_score = self.first_pass_score(event)

        # Look at nearby events.
        left = max(0, event.event_id - 5)
        right = min(
            len(events),
            event.event_id + 6
        )

        nearby = events[left:right]

        same_source_events = [
            other
            for other in nearby
            if (
                other.source == event.source
                and other.event_id != event.event_id
            )
        ]

        # Is there supporting evidence nearby?
        if same_source_events:

            corroboration = max(
                x.novelty
                for x in same_source_events
            )

        else:

            corroboration = 0.0

        # Context can strengthen attention.
        context_boost = 0

        if event.context == self.active_context:
            context_boost = 0.15

        score = (
            original_score
            + 0.18 * corroboration
            + context_boost
        )

        return min(score, 1.0)

    # --------------------------------------------------------
    # EVENT SELECTION
    # --------------------------------------------------------

    def select(self, events):

        # Reserve some processing capacity for rechecking.

        first_budget = round(
            self.processing_budget
            * (1 - self.recheck_fraction)
        )

        recheck_budget = (
            self.processing_budget
            - first_budget
        )

        # -----------------------------
        # Fast filtering
        # -----------------------------

        ranked = sorted(
            events,
            key=self.first_pass_score,
            reverse=True
        )

        selected = ranked[:first_budget]

        # -----------------------------
        # Deferred attention
        # -----------------------------

        remaining = ranked[
            first_budget:
            first_budget + self.processing_budget * 3
        ]

        # -----------------------------
        # Deliberate recheck
        # -----------------------------

        reconsidered = sorted(
            remaining,
            key=lambda e:
                self.recheck_score(e, events),
            reverse=True
        )

        selected.extend(
            reconsidered[:recheck_budget]
        )

        # Never exceed processing budget.

        return selected[
            :self.processing_budget
        ]


# ============================================================
# PROCEDURAL MEMORY
# ============================================================

class RuleMemory:

    def __init__(self):

        self.rules = []

    def add_rule(
        self,
        name,
        condition,
        action
    ):

        self.rules.append(
            (
                name,
                condition,
                action
            )
        )

    def execute(self, event):

        responses = []

        for name, condition, action in self.rules:

            if condition(event):

                responses.append(
                    {
                        "rule": name,
                        "action": action(event)
                    }
                )

        return responses


# ============================================================
# BUILD BASIC "LEARNED" PROCEDURES
# ============================================================

def build_memory():

    memory = RuleMemory()

    # Rule 1

    memory.add_rule(

        "ObstacleRule",

        lambda event:
            event.source == "motion"
            and event.value > 0.72,

        lambda event:
            "Possible obstacle detected"
    )

    # Rule 2

    memory.add_rule(

        "VisualChangeRule",

        lambda event:
            event.source == "vision"
            and
            event.context == "navigation"
            and
            event.novelty > 0.43,

        lambda event:
            "Unexpected visual change"
    )

    # Rule 3

    memory.add_rule(

        "AudioAlertRule",

        lambda event:
            event.source == "audio"
            and event.value > 0.88,

        lambda event:
            "Strong audio event"
    )

    return memory


# ============================================================
# SIMULATED ENVIRONMENT
# ============================================================

def generate_events(
    number_of_events=120,
    seed=7
):

    random.seed(seed)

    sources = [
        "vision",
        "audio",
        "motion",
        "temperature"
    ]

    contexts = [
        "navigation",
        "conversation",
        "idle"
    ]

    previous_values = {
        source: 0.10
        for source in sources
    }

    events = []

    for event_id in range(
        number_of_events
    ):

        source = random.choice(sources)

        context = random.choice(contexts)

        value = random.random()

        previous = previous_values[source]

        previous_values[source] = value

        novelty = abs(
            value - previous
        )

        # ====================================================
        # Hidden experimental ground truth
        #
        # The filter DOES NOT receive this information.
        # It is only used to evaluate whether the system
        # successfully detected relevant information.
        # ====================================================

        important = (

            (
                source == "motion"
                and value > 0.72
            )

            or

            (
                source == "vision"
                and context == "navigation"
                and novelty > 0.43
            )

            or

            (
                source == "audio"
                and context == "conversation"
                and value > 0.88
            )

            or

            (
                source == "temperature"
                and value > 0.96
            )
        )

        events.append(

            Event(

                event_id=event_id,

                source=source,

                value=value,

                previous_value=previous,

                context=context,

                important=important
            )
        )

    return events


# ============================================================
# EVALUATION
# ============================================================

def evaluate(
    selected,
    all_events
):

    important_ids = {

        event.event_id

        for event in all_events

        if event.important
    }

    selected_ids = {

        event.event_id

        for event in selected
    }

    true_positive = len(
        important_ids
        & selected_ids
    )

    false_positive = len(
        selected_ids
        - important_ids
    )

    false_negative = len(
        important_ids
        - selected_ids
    )

    if (
        true_positive
        + false_negative
    ):

        recall = (

            true_positive
            /
            (
                true_positive
                + false_negative
            )
        )

    else:

        recall = 0

    if (
        true_positive
        + false_positive
    ):

        precision = (

            true_positive
            /
            (
                true_positive
                + false_positive
            )
        )

    else:

        precision = 0

    return {

        "processed":
            len(selected_ids),

        "important_detected":
            true_positive,

        "recall":
            recall,

        "precision":
            precision
    }


# ============================================================
# BENCHMARK
# ============================================================

def benchmark(
    trials=200,
    budget=20
):

    fixed_recalls = []

    hi_recalls = []

    fixed_precisions = []

    hi_precisions = []

    for seed in range(trials):

        events = generate_events(
            seed=seed
        )

        # -------------------------------
        # Fixed architecture
        # -------------------------------

        fixed = FixedFilter(
            processing_budget=budget
        )

        fixed_selected = (
            fixed.select(events)
        )

        fixed_result = evaluate(
            fixed_selected,
            events
        )

        # -------------------------------
        # Humanoid Intelligence
        # -------------------------------

        humanoid = HumanoidFilter(
            processing_budget=budget,
            recheck_fraction=0.30
        )

        hi_selected = (
            humanoid.select(events)
        )

        hi_result = evaluate(
            hi_selected,
            events
        )

        fixed_recalls.append(
            fixed_result["recall"]
        )

        hi_recalls.append(
            hi_result["recall"]
        )

        fixed_precisions.append(
            fixed_result["precision"]
        )

        hi_precisions.append(
            hi_result["precision"]
        )

    print(
        "\n======== EXPERIMENT ========"
    )

    print(
        "Trials:",
        trials
    )

    print(
        "Processing budget:",
        budget
    )

    print()

    print(
        "FIXED FILTER"
    )

    print(
        "Average recall:",
        round(
            statistics.mean(
                fixed_recalls
            ),
            3
        )
    )

    print(
        "Average precision:",
        round(
            statistics.mean(
                fixed_precisions
            ),
            3
        )
    )

    print()

    print(
        "HUMANOID INTELLIGENCE"
    )

    print(
        "Average recall:",
        round(
            statistics.mean(
                hi_recalls
            ),
            3
        )
    )

    print(
        "Average precision:",
        round(
            statistics.mean(
                hi_precisions
            ),
            3
        )
    )


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    benchmark(
        trials=200,
        budget=20
    )


======== EXPERIMENT ========
Trials: 200
Processing budget: 20

FIXED FILTER
Average recall: 0.548
Average precision: 0.373

HUMANOID INTELLIGENCE
Average recall: 0.589
Average precision: 0.4
